# Architecture Tour: GPT → LLaMA → Hybrid

Walk through three architectures side-by-side. For each:
config → model → forward pass → parameter counting.

**Prerequisites:** `uv sync --extra dev`

In [ ]:
import mlx.core as mx
import mlx.utils

from lmxlab.models.base import LanguageModel
from lmxlab.models.gpt import gpt_tiny
from lmxlab.models.llama import llama_tiny
from lmxlab.models.falcon import falcon_h1_tiny

## 1. GPT: The Classic Transformer

GPT uses: LayerNorm, standard MHA, standard FFN (GELU),
sinusoidal position encoding, pre-norm, bias everywhere.

In [ ]:
gpt_cfg = gpt_tiny()
print(f"Block config:")
print(f"  Attention: {gpt_cfg.block.attention}")
print(f"  FFN:       {gpt_cfg.block.ffn}")
print(f"  Norm:      {gpt_cfg.block.norm}")
print(f"  Position:  {gpt_cfg.block.position}")
print(f"  d_model:   {gpt_cfg.block.d_model}")
print(f"  n_heads:   {gpt_cfg.block.n_heads}")
print(f"  d_ff:      {gpt_cfg.block.d_ff}")
print(f"  Bias:      {gpt_cfg.block.bias}")
print(f"\nModel config:")
print(f"  n_layers:  {gpt_cfg.n_layers}")
print(f"  vocab:     {gpt_cfg.vocab_size}")

In [ ]:
gpt = LanguageModel(gpt_cfg)
mx.eval(gpt.parameters())
print(f"GPT params: {gpt.count_parameters():,}")

# Forward pass
tokens = mx.zeros((1, 16), dtype=mx.int32)
logits, _ = gpt(tokens)
mx.eval(logits)
print(f"Input:  {tokens.shape}  (batch, seq_len)")
print(f"Output: {logits.shape}  (batch, seq_len, vocab)")

## 2. LLaMA: Modern Best Practices

LLaMA uses: RMSNorm, GQA (fewer KV heads), gated FFN (SwiGLU),
RoPE position encoding, no bias.

In [ ]:
llama_cfg = llama_tiny()
print(f"LLaMA differences from GPT:")
print(f"  Attention: {llama_cfg.block.attention} (GQA with {llama_cfg.block.n_kv_heads} KV heads)")
print(f"  FFN:       {llama_cfg.block.ffn} (SwiGLU — 3 projections)")
print(f"  Norm:      {llama_cfg.block.norm} (no mean subtraction)")
print(f"  Position:  {llama_cfg.block.position} (rotary embeddings)")
print(f"  Bias:      {llama_cfg.block.bias} (removed entirely)")

In [ ]:
llama = LanguageModel(llama_cfg)
mx.eval(llama.parameters())
print(f"LLaMA params: {llama.count_parameters():,}")

logits, _ = llama(tokens)
mx.eval(logits)
print(f"Output: {logits.shape}")

## 3. Falcon-H1: SSM/Attention Hybrid

Falcon-H1 interleaves Mamba-2 (SSM) and GQA layers in a
MMM\* pattern: 3 Mamba blocks for every 1 attention block.
SSM layers have O(n) complexity vs O(n²) for attention.

In [ ]:
falcon_cfg = falcon_h1_tiny()
print(f"Falcon-H1 layer types:")
for i in range(falcon_cfg.n_layers):
    bcfg = falcon_cfg.get_block_config(i)
    layer_type = "SSM (Mamba-2)" if "mamba" in bcfg.attention else "Attention (GQA)"
    print(f"  Layer {i}: {layer_type}")

In [ ]:
falcon = LanguageModel(falcon_cfg)
mx.eval(falcon.parameters())
print(f"Falcon-H1 params: {falcon.count_parameters():,}")

logits, _ = falcon(tokens)
mx.eval(logits)
print(f"Output: {logits.shape}")

## Parameter Breakdown

Let's compare where the parameters are in each architecture.

In [ ]:
for name, model in [("GPT", gpt), ("LLaMA", llama), ("Falcon-H1", falcon)]:
    total = model.count_parameters()
    embed = model.embed.weight.size
    blocks = sum(
        p.size
        for _, p in mlx.utils.tree_flatten(
            [b.parameters() for b in model.blocks]
        )
    )
    print(f"{name}: {total:,} total = {embed:,} embed + {blocks:,} blocks")